# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsiddan25/program/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
%pip -q install duckdb huggingface_hub

import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": (
        f"read_parquet('{REL}/dim_clients.parquet')"
    ),
    "dim_content": (
        f"read_parquet('{REL}/dim_content.parquet')"
    ),
    "fact_daily": (
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
    ),
    "fact_daily_march": (
        f"read_parquet("
        f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
        f")"
    ),
    "fact_query_90d": (
        f"read_parquet('{REL}/fact_content_query_90d.parquet')"
    ),
}

print("DuckDB connection and table paths are ready.")

Paste your Hugging Face READ token (hf_...): ··········
DuckDB connection and table paths are ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one pseudonymized client, content page, and report date, from March 1 through March 31, 2026. I will aggregate these daily rows so that one row in the analysis table represents one content page for one pseudonymized client, summarized over March 2026

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_check = con.sql(f"""
    WITH march AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id
        FROM {TABLES["fact_daily_march"]}
    ),
    duplicate_groups AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) AS row_count
        FROM march
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
    SELECT
        (SELECT COUNT(*) FROM march) AS daily_rows,
        (
            SELECT COUNT(*)
            FROM (
                SELECT DISTINCT client_hash_id, content_hash_id
                FROM march
            )
        ) AS distinct_client_content_pages,
        (SELECT MIN(report_date) FROM march) AS start_date,
        (SELECT MAX(report_date) FROM march) AS end_date,
        (SELECT COUNT(*) FROM duplicate_groups) AS duplicate_grain_groups
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,daily_rows,distinct_client_content_pages,start_date,end_date,duplicate_grain_groups
0,9841378,331437,2026-03-01,2026-03-31,0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I will use the dim_content and fact_daily_performance mainly. I will use fact_query_90_d in the future. The features I will use are impressions, avg_position, CTR, content_age_days, days_since_last_update.

The label is predicting which pages decline, but this is a proxy. This is used to access which pages most likely need a refresh.

For context, I will say that client_hash_id is useful as it groups pages by client and supports client-level validation. Content_hash_id identifies and joins content pages. Report_date defines the measurement window, and
content_type supports grouped checks and interpretation.

What I will purposely exclude is url_hash_id and keyword_hash_id. I will also not include optimization_eligible_date and last_optimized_date as these may encode an existing product decision and create circular results

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

daily_fields = con.sql(
    f"DESCRIBE SELECT * FROM {TABLES['fact_daily_march']}"
).df()

content_fields = con.sql(
    f"DESCRIBE SELECT * FROM {TABLES['dim_content']}"
).df()

display(daily_fields)
display(content_fields)


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

features = con.sql(f"""
    WITH march_aggregated AS (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(DISTINCT report_date) AS days_observed,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_sum_position) AS sum_position
        FROM {TABLES["fact_daily_march"]}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.days_observed,

        m.impressions,

        CASE
            WHEN m.impressions > 0
            THEN m.sum_position * 1.0 / m.impressions
        END AS avg_position,

        CASE
            WHEN m.impressions > 0
            THEN 100.0 * m.clicks / m.impressions
        END AS ctr,

        DATE_DIFF(
            'day',
            d.content_created_date,
            DATE '2026-03-31'
        ) AS content_age_days,

        DATE_DIFF(
            'day',
            d.content_updated_date,
            DATE '2026-03-31'
        ) AS days_since_last_update,

        d.word_count

    FROM march_aggregated AS m
    LEFT JOIN {TABLES["dim_content"]} AS d
        ON m.client_hash_id = d.client_hash_id
       AND m.content_hash_id = d.content_hash_id
""").df()

print(f"Analysis rows: {len(features):,}")
print(f"Analysis columns: {features.shape[1]}")

verification = {
    "total_rows": len(features),
    "duplicate_client_content_rows": features.duplicated(
        subset=["client_hash_id", "content_hash_id"]
    ).sum(),
    "minimum_days_observed": features["days_observed"].min(),
    "maximum_days_observed": features["days_observed"].max(),
    "zero_impression_pages": (features["impressions"] == 0).sum(),
}

verification

columns_to_check = [
    "impressions",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

missing_check = (
    features[columns_to_check]
    .isna()
    .sum()
    .to_frame(name="missing_count")
)

missing_check["missing_percent"] = (
    100 * missing_check["missing_count"] / len(features)
).round(2)

missing_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Analysis rows: 176,738
Analysis columns: 9


,missing_count,missing_percent
impressions,0,0.0
avg_position,0,0.0
ctr,0,0.0
content_age_days,0,0.0
days_since_last_update,0,0.0
word_count,55315,31.3


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data cannot provide a complete and equally long history for every client or page because their data begins on different dates. Some early rows contain GSC data but no available GA4 data, so zero-filled GA4 values during those periods cannot be interpreted as zero engagement. The query-level table uses a fixed 90-day window that may overlap with the period used to define the label, which could cause data leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.